**Project Objective:**


To create a **data-driven** YouTube channel and supporting platforms, **producing lo-fi, binaural beat, and study-focused** content by **integrating live-recorded drum performances with AI-generated melodic, harmonic, and ambient layers** within a collaborative **human–AI music production workflow**, while applying **machine learning models** to analyze performance patterns within an API-sampled dataset of LoFi-related and other selected content.

<img src="neurobeat_logo.jpg" alt="description" width="400"/>

### 🎯 Project Scope & Methodology

The goal is to analyze and predict engagement dynamics within highly specific content niches: **Lofi, Chillhop, and Study-oriented media**. We prioritize **interpretability and feature relevance** to understand what truly drives audience interaction beyond mere scale.

The model predicts **Expected Audience Engagement** using a combination of content-specific triggers, temporal trends, and channel authority features extracted from the **YouTube Data API.**

**Engagement (Likes + Comments)** was selected as the core metric because it represents active user commitment, making it more appropriate for **early-stage or niche content** where community depth matters more than raw view counts.

---
# NeuroBeats – Applied Machine Learning for YouTube Niche Engagement

## Project Overview

NeuroBeats is an end-to-end applied Machine Learning project focused on modeling engagement dynamics within highly specific YouTube niches:

- Lofi
- Chillhop
- Study-oriented content

The objective was not only to build a predictive model, but to:

- Engineer domain-specific features
- Detect and correct data leakage
- Design robust validation strategies
- Prioritize generalization over inflated metrics
- Interpret model behavior using feature importance tools

This project emphasizes ML rigor over surface-level analytics.

---

## Business Framing

Can we predict expected audience engagement using structured metadata, niche-specific triggers, temporal patterns, and channel authority signals?

Rather than modeling views, the project targets:

**Engagement = Likes + Comments**

Engagement was chosen because it reflects active audience commitment — especially relevant in niche communities.

Target transformation:
`log1p(likes + comments)`  
to normalize skewed distributions and reduce viral outlier distortion.

---

## Feature Engineering (Core Contribution)

Significant effort was invested in domain-driven feature creation.

### Content & Niche Strategy (Engineered)
- `is_pomodoro`
- `is_live`
- `is_long_form` (>20 minutes)
- `has_visuals`
- `video_duration_seconds`
- `title_length`

These features capture behavioral triggers specific to the Lofi/Study ecosystem.

### Temporal Modeling
- `year`
- `year_label`

Captures post-2020 niche acceleration and structural shifts.

### Channel Authority Metrics
- `log_subscribers`
- `log_channel_views`
- `channel_video_count`

Log transformations were applied to stabilize variance and reduce heteroscedasticity.

---

## Model Development & Validation

### Baseline Phase
- Linear Regression
- Random Forest

Initial Random Split produced R² ≈ 0.49  
→ Identified as leakage-prone and overly optimistic.

### Leakage Detection & Correction

Evaluation transitioned to:

- GroupShuffleSplit (by channel)
- GroupKFold (by keyword)

This forced cross-niche generalization and removed hidden signal leakage.

### Final Model Comparison

| Model | Validation Strategy | R² | Notes |
|-------|--------------------|-----|------|
| RF | Random Split | 0.49 | Leakage likely |
| RF | GroupKFold | 0.18 | Realistic baseline |
| XGBoost | GroupKFold | 0.20–0.37 | Strongest generalizable model |

Peak fold performance reached R² = 0.83 in niche-specific segments.

---

## Feature Importance & Model Interpretation

Interpretability was prioritized over raw performance.

- SHAP Global Explainer used for impact analysis
- Feature importance showed:

  - Long-form duration strongly influences engagement
  - Channel authority stabilizes predictions
  - Keyword stacking has limited predictive impact
  - Format-driven triggers add measurable signal

The project demonstrates how domain-aligned feature engineering can outperform superficial metadata optimization.

---

## Technical Stack

- Python
- Pandas
- Scikit-learn
- XGBoost
- SHAP
- YouTube Data API
- Cross-validation (GroupKFold)
- REAPER (applied production layer)

---

## Engineering Highlights

- End-to-end ML pipeline ownership
- Advanced feature engineering
- Leakage detection & mitigation
- Group-aware cross-validation
- Model stability analysis
- SHAP-based interpretability
- Translation of ML insights into real production decisions

---

## What This Project Demonstrates

- Applied ML thinking
- Model robustness prioritization
- Evaluation discipline
- Feature design in low-signal environments
- Practical ML implementation beyond notebook-level experimentation

---

## Author

David Hernandez  
Applied ML & Data Analyst  
Open to ML Engineer / Data-focused roles

### 1) Library and Module import

In [ ]:
# Installing libraries

# !pip install google-api-python-client


In [ ]:
from googleapiclient.discovery import build
import pandas as pd
import json
import time

### 2) API, YouTube Object and DF Definitions

In [ ]:
# Note: API key should be kept secret, and not hardcoded in production code
# Create a file and save it as .env, in it write: API_KEY=your_api_key_here
# Install the python-dotenv package if you want to load environment variables from a .env file

import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_KEY")

In [ ]:
# Building the YouTube service object

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY
)

# Not having any errors up to this point indicates the API Key is valid

In [ ]:
# 1) Define keywords and date ranges for the search - Beginning of the code cell for configuration

# Single code cell, labeled by purpose

# ======================
# CONFIG
# ======================

KEYWORDS = [
    "lofi",
    "lofi chill hop",
    "chillhop",
    "binaural beats",
    "study with me",
]

DATE_RANGES = {
    "2019_2026": ("2019-01-01T00:00:00Z", "2026-12-31T23:59:59Z")
}


# ======================
# SEARCH FUNCTION
# ======================

def search_videos(keyword, published_after, published_before, max_total=200):
    videos = []
    next_page_token = None

    while len(videos) < max_total:
        request = youtube.search().list(
            q=keyword,
            part="snippet",
            type="video",
            publishedAfter=published_after,
            publishedBefore=published_before,
            maxResults=50,
            pageToken=next_page_token
        )

        response = request.execute()

        for item in response.get("items", []):
            videos.append({
                "video_id": item["id"]["videoId"],
                "title": item["snippet"]["title"],
                "description": item["snippet"]["description"],
                "published_at": item["snippet"]["publishedAt"],
                "channel_title": item["snippet"]["channelTitle"],
                "keyword_used": keyword
            })

        next_page_token = response.get("nextPageToken")

        if not next_page_token:
            break

    return pd.DataFrame(videos)


# ======================
# STATS FUNCTION
# ======================

def get_video_stats(video_ids): # This funcion consumes API quota
    request = youtube.videos().list(
        part="statistics,contentDetails",
        id=",".join(video_ids)
    )
    response = request.execute()

    stats = []
    for item in response.get("items", []):
        stats.append({
            "video_id": item["id"],
            "view_count": int(item["statistics"].get("viewCount", 0)),
            "like_count": int(item["statistics"].get("likeCount", 0)),
            "comment_count": int(item["statistics"].get("commentCount", 0)),
            "duration": item.get("contentDetails", {}).get("duration", None)
        })

    return pd.DataFrame(stats)

# ======================
# FULL PIPELINE
# ======================

all_dfs = []

for period, (start_date, end_date) in DATE_RANGES.items():
    for keyword in KEYWORDS:
        df_temp = search_videos(keyword, start_date, end_date, max_total=200)

        if not df_temp.empty:
            df_temp["period"] = period
            all_dfs.append(df_temp)

# Combine search results
df_all = pd.concat(all_dfs, ignore_index=True)


# Get stats in batches of 50 (YouTube API limit)
video_ids = df_all["video_id"].unique().tolist()

all_stats = []

for i in range(0, len(video_ids), 50): 
    batch = video_ids[i:i+50]
    batch_df = get_video_stats(batch)
    all_stats.append(batch_df)

df_stats = pd.concat(all_stats, ignore_index=True)


# Final dataset
df_full = df_all.merge(df_stats, on="video_id", how="left")

df_full.head()


In [ ]:
df_full.shape

In [ ]:
df_full = df_full[df_full["view_count"] > 100].copy()


In [ ]:
df_stats.shape

In [ ]:
df_stats.columns
# These videos will suffice our need, our target is the level of engagement which comes from comments and likes, not the number of videos themselves.

In [ ]:
df_full.columns

In [ ]:
features = [
    "video_duration_seconds",
    "title_length",
    "year",
    "keyword_used"
]


In [ ]:
X = pd.get_dummies(
    df_full[features],
    columns=["keyword_used"],
    drop_first=True
)

y = df_full["engagement_per_view"]


In [ ]:
# 2) Train-test split
from sklearn.model_selection import train_test_split

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [ ]:
# Training model

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

rf_random = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_random.fit(X_train_r, y_train_r)

y_pred_r = rf_random.predict(X_test_r)

r2_random = r2_score(y_test_r, y_pred_r)
mae_random = mean_absolute_error(y_test_r, y_pred_r)

print("Random Split Results")
print("R2:", r2_random)
print("MAE:", mae_random)


In [ ]:
# Cross-validation
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    rf_random,
    X,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("Cross-Validation R2 Mean:", cv_scores.mean())
print("Cross-Validation R2 Std:", cv_scores.std())


In [ ]:
# Feature importance
import pandas as pd

importances = pd.Series(
    rf_random.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importances)


In [ ]:
# Getting channel_id for Group Split (Consumes API quota)
def get_video_channels(video_ids):
    rows = []

    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]

        request = youtube.videos().list(
            part="snippet",
            id=",".join(batch)
        )
        response = request.execute()

        for item in response.get("items", []):
            rows.append({
                "video_id": item["id"],
                "channel_id": item["snippet"]["channelId"]
            })

    return pd.DataFrame(rows)


# Execute
video_ids = df_full["video_id"].unique().tolist()
df_video_channels = get_video_channels(video_ids)

# Merge
df_full = df_full.merge(df_video_channels, on="video_id", how="left")


In [ ]:
# 3) GroupShuffleSplit

from sklearn.model_selection import GroupShuffleSplit

groups = df_full["channel_id"]

gss = GroupShuffleSplit(
    test_size=0.2,
    n_splits=1,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_g = X.iloc[train_idx]
X_test_g  = X.iloc[test_idx]
y_train_g = y.iloc[train_idx]
y_test_g  = y.iloc[test_idx]


In [ ]:
# 4) Train model on Group Split

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

rf_group = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_group.fit(X_train_g, y_train_g)

y_pred_g = rf_group.predict(X_test_g)

r2_group = r2_score(y_test_g, y_pred_g)
mae_group = mean_absolute_error(y_test_g, y_pred_g)

print("Group Split Results")
print("R2:", r2_group)
print("MAE:", mae_group)


In [ ]:
# Feature importance for Group Spli
import pandas as pd

importances_group = pd.Series(
    rf_group.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importances_group)


In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "RF (Random Split)",
        "RF (Group Split)"
    ],
    "R2": [
        r2_random,
        r2_group
    ],
    "MAE": [
        mae_random,
        mae_group
    ]
})

comparison


Based on the results we are going to predict another target, stepping away from engagement per view and moving into log engagement. 
From this point onwards we start our journey through the creation, training and evaluation of several ML models. Numbered cells to navigate through the notebook easily. 

In [ ]:
# 1) Creating new feature: log_engagement

import numpy as np

# Create raw engagement
df_full["engagement"] = df_full["like_count"] + df_full["comment_count"]

# Log transform
df_full["log_engagement"] = np.log1p(df_full["engagement"])


In [ ]:
#2 ) Defining features

features = [
    "video_duration_seconds",
    "title_length",
    "year",
    "keyword_used"
]

X = pd.get_dummies(
    df_full[features],
    columns=["keyword_used"],
    drop_first=True
)

y = df_full["log_engagement"]


In [ ]:
# 3) Train test split for newly created log_engagement target

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# 4) Creating RF Model

from sklearn.ensemble import RandomForestRegressor

rf_log = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_log.fit(X_train, y_train)


In [ ]:
# 5) Evaluating model

from sklearn.metrics import r2_score, mean_absolute_error

y_pred = rf_log.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))


In [ ]:
# 6) Cross-Validation

from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    rf_log,
    X,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("Mean CV R2:", scores.mean())
print("Std CV R2:", scores.std())
print("All folds:", scores)


In [ ]:
# 7) Putting MAE into perspective by comparing it to the mean engagement rate
print("Mean log_engagement:", df_full["log_engagement"].mean())
print("Std log_engagement:", df_full["log_engagement"].std())


In [ ]:
# 8) Convert predictions back to original scale
y_pred_original = np.expm1(y_pred)
y_test_original = np.expm1(y_test)

mae_original = mean_absolute_error(y_test_original, y_pred_original)
print("MAE in original engagement units:", mae_original)


In [ ]:
# 9) Group Shuffle Split with log_engagement target

from sklearn.model_selection import GroupShuffleSplit

# Defining groups based on the original 'keyword_used' column 
groups = df_full["keyword_used"]

# Configuring split by groups
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)


# Getting indexes and spliting
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


In [ ]:
# 10) Creating the model

from sklearn.ensemble import RandomForestRegressor

rf_log = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_log.fit(X_train, y_train)

In [ ]:
# 11) Evaluating model

from sklearn.metrics import r2_score, mean_absolute_error

y_pred = rf_log.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))


In [ ]:
# 12) Cross validation for GSS

from sklearn.model_selection import cross_val_score, GroupKFold

# Cross-Validation respecting groups
group_kfold = GroupKFold(n_splits=5)

scores = cross_val_score(
    rf_log,
    X,
    y,
    cv=group_kfold,        #Changing object to GroupKFold
    groups=df_full["keyword_used"],
    scoring="r2",
    n_jobs=-1
)

print("Mean CV R2 (Realistic):", scores.mean())
print("Std CV R2:", scores.std())
print("All folds:", scores)


In [ ]:
# 13) Comparison table

import pandas as pd

final_comparison = pd.DataFrame({
    "Model": [
        "Random Forest (log target)",
        "Random Forest (log target)",
        "Random Forest (log target)",
        "Random Forest (log target)"
    ],
    "Evaluation Strategy": [
        "Random Train/Test Split",
        "Random Split + Cross Validation",
        "GroupShuffleSplit (by channel)",
        "GroupKFold CV (by keyword)"
    ],
    "R2": [
        0.4938428394835267,     # optimistic split
        -0.4627338698616225,    # random CV
        0.010204283392430069,   # GSS single split
        0.18673494861529644     # NEW realistic CV
    ],
    "R2 Std (CV)": [
        None,
        1.1906873711128545,
        None,
        0.279440865459969
    ],
    "Notes": [
        "Optimistic – leakage likely",
        "Unstable performance",
        "Leakage controlled (single split)",
        "Most realistic and stable estimate"
    ]
})

final_comparison


Initial random splits showed optimistic performance **(R² ≈ 0.49),** but cross-validation revealed instability due to **potential channel leakage**.
After implementing a **GroupShuffleSplit by channel_id, the model achieved a more realistic R² ≈ 0.17**, indicating moderate predictive signal while avoiding data leakage.

In [ ]:
# 14) Bar chart

import matplotlib.pyplot as plt
import pandas as pd

# Data for visualization
models = [
    "Random Split",
    "Random CV",
    "GroupShuffleSplit",
    "GroupKFold CV (Final)"
]

r2_scores = [
    0.4938428394835267,
    -0.4627338698616225,
    0.010204283392430069,
    0.18673494861529644
]

# Create dataframe
df_plot = pd.DataFrame({
    "Model": models,
    "R2": r2_scores
})

# Plot
plt.figure(figsize=(10,6))
plt.bar(df_plot["Model"], df_plot["R2"])

plt.axhline(0)  # zero baseline
plt.title("Model Performance Comparison (R2)")
plt.ylabel("R2 Score")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# 15) Feature importance for RF model

import pandas as pd
import matplotlib.pyplot as plt

# 1) FIT THE MODEL ONE LAST TIME
# We use the full X and y to get the most robust importance
rf_log.fit(X, y)

# 2) EXTRACT FEATURE IMPORTANCES
importances = rf_log.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})

# 3) SORT AND FILTER
# Since keywords (dummies) might clutter the plot, let's see the Top 15
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)

# 4) VISUALIZATION
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['feature'].head(15), feature_importance_df['importance'].head(15), color='skyblue')
plt.xlabel("Importance Score")
plt.title("Top 15 Features - Random Forest Importance")
plt.gca().invert_yaxis()
plt.show()

# 5) ANALYZE KEYWORD IMPACT vs METRICS
non_keyword_features = ["video_duration_seconds", "title_length", "year"]
core_importance = feature_importance_df[feature_importance_df['feature'].isin(non_keyword_features)]
print("\n--- Core Features Importance ---")
print(core_importance)


In [ ]:
# 16) Introducing MLP: RF + Stacking Regressor (No Cross-Validation for simplicity after several failed attempts)
from sklearn.metrics import mean_absolute_error

# 1) PREPARE THE LIST FOR MAE SCORES
manual_mae_scores = []

print("Starting Stacking Loop for MAE calculation...")

# 2) RE-RUNNING THE FOLDS (Simplified for MAE)
for i, (train_idx, val_idx) in enumerate(gkfold.split(X, y, groups=df_full["keyword_used"])):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # Define and fit
    stacking_model = StackingRegressor(
        estimators=[('rf', rf_component), ('mlp', mlp_component)],
        final_estimator=RidgeCV(),
        cv=5,
        n_jobs=-1
    )
    stacking_model.fit(X_train_fold, y_train_fold)
    
    # Predict and calculate MAE
    fold_pred = stacking_model.predict(X_val_fold)
    fold_mae = mean_absolute_error(y_val_fold, fold_pred)
    
    manual_mae_scores.append(fold_mae)
    print(f"Fold {i+1} completed. MAE: {fold_mae:.4f}")

# 3) FINAL MAE SUMMARY
manual_mae_scores = np.array(manual_mae_scores)
print("\n--- Final MAE Results ---")
print(f"Mean CV MAE: {manual_mae_scores.mean():.4f}")
print(f"All Folds MAE: {manual_mae_scores}")


In [ ]:
# 17) Introducing MLP: RF + Stacking Regressor (No Cross-Validation for simplicity after several failed attempts)
# R2 Calculations

from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
import numpy as np

# 1) PREPARE THE MANUAL CROSS-VALIDATION
gkfold = GroupKFold(n_splits=5)
manual_scores = []

print("Starting Final Stacking Loop (Simplified CV)...")

for i, (train_idx, val_idx) in enumerate(gkfold.split(X, y, groups=df_full["keyword_used"])):
    
    # Split data
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # 2) DEFINE STACKING MODEL
    # We use cv='prefit' or a simple integer to avoid the 'groups' requirement in fit()
    # This tells the model to use a standard 5-fold split for the meta-learner
    stacking_model = StackingRegressor(
        estimators=[('rf', rf_component), ('mlp', mlp_component)],
        final_estimator=RidgeCV(),
        cv=5, # Using a standard integer avoids the metadata routing error
        n_jobs=-1
    )
    
    # 3) FIT AND EVALUATE (No 'groups' parameter here to avoid errors)
    stacking_model.fit(X_train_fold, y_train_fold)
    
    fold_pred = stacking_model.predict(X_val_fold)
    fold_r2 = r2_score(y_val_fold, fold_pred)
    
    manual_scores.append(fold_r2)
    print(f"Fold {i+1} completed. R2: {fold_r2:.4f}")

# 4) FINAL SUMMARY
manual_scores = np.array(manual_scores)
print("\n--- Final Results ---")
print(f"Mean CV R2: {manual_scores.mean():.4f}")
print(f"All Folds: {manual_scores}")



In [ ]:
!pip install xgboost


In [ ]:
#18) Introducing XGBoost with metadata routing (GroupKFold) for a more realistic evaluation

import xgboost as xgb

# 1) PREPARE METADATA ROUTING
# Since we enabled metadata_routing before, we must pass groups inside a 'params' dictionary
fit_params = {"groups": groups_niche}

# 2) DEFINE XGBOOST MODEL (With niche features)
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# 3) RUN CROSS-VALIDATION USING THE NEW SYNTAX
# We pass 'params' instead of 'groups' directly to satisfy the metadata routing
scores_xgb = cross_val_score(
    xgb_model,
    X_niche,
    y_niche,
    cv=gkfold,
    params=fit_params, # <--- THIS FIXES THE VALUEERROR
    scoring="r2",
    n_jobs=-1
)

# 4) RESULTS SUMMARY
print(f"XGBoost + Niche Features Mean R2: {scores_xgb.mean():.4f}")
print(f"All Folds R2: {scores_xgb}")


In [ ]:
# 19) Updated comparison table with XGBoost results

import pandas as pd

# Adding XGBoost results to your comparison table
new_row = {
    "Model": "XGBoost (Niche Features)",
    "Evaluation Strategy": "GroupKFold CV (by keyword)",
    "R2": 0.2039, 
    "R2 Std (CV)": 0.4337, # High variance but higher mean
    "Notes": "Best performer - Captures Lofi/Study niche logic"
}

# Update the DataFrame
final_comparison = pd.concat([final_comparison, pd.DataFrame([new_row])], ignore_index=True)

# Display the updated table
final_comparison


In [ ]:
# 20) Creating a function to get more stats from API
# Running this cell consumes API quota

def get_channel_stats(channel_ids):
    # Removing duplicates to save API quota
    unique_channels = list(set(channel_ids))
    all_channel_data = []

    # API allows 50 IDs per request
    for i in range(0, len(unique_channels), 50):
        request = youtube.channels().list(
            part="statistics",
            id=",".join(unique_channels[i:i+50])
        )
        response = request.execute()

        for item in response.get("items", []):
            all_channel_data.append({
                "channel_id": item["id"],
                "subscriber_count": int(item["statistics"].get("subscriberCount", 0)),
                "channel_view_count": int(item["statistics"].get("viewCount", 0)),
                "channel_video_count": int(item["statistics"].get("videoCount", 0))
            })
    
    return pd.DataFrame(all_channel_data)


In [ ]:
# Check exact column names in your dataframe
print(df_full.columns.tolist())


In [ ]:
# 21) Merging channel stats back to main dataframe

# 1) FETCH THE DATA
# We call the function and store the result in 'df_channels'
channel_ids_list = df_full["channel_id"].unique().tolist()
df_channels = get_channel_stats(channel_ids_list)

# 2) MERGE WITH YOUR MAIN DATASET
# This adds the subscriber_count, etc., to every video row
df_full = df_full.merge(df_channels, on="channel_id", how="left")

# 3) DATA CLEANING (Handle missing values if any)
df_full["subscriber_count"] = df_full["subscriber_count"].fillna(0)
df_full["channel_view_count"] = df_full["channel_view_count"].fillna(0)
df_full["channel_video_count"] = df_full["channel_video_count"].fillna(0)

# 4) LOG TRANSFORMATION
# Essential for large numbers like subscriber counts
df_full["log_subscribers"] = np.log1p(df_full["subscriber_count"])
df_full["log_channel_views"] = np.log1p(df_full["channel_view_count"])

# 5) RUN THE ULTIMATE XGBOOST
features_ultimate = [
    "video_duration_seconds", "title_length", "year", 
    "is_live", "is_pomodoro", "has_visuals", "is_long_form",
    "log_subscribers", "log_channel_views", "channel_video_count"
]

X_final = df_full[features_ultimate]
y_final = df_full["log_engagement"]
groups_final = df_full["keyword_used"]

xgb_final = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.02, max_depth=6, random_state=42, n_jobs=-1)

final_scores = cross_val_score(
    xgb_final, X_final, y_final,
    cv=gkfold, params={"groups": groups_final},
    scoring="r2", n_jobs=-1
)

print(f"ULTIMATE XGBOOST Mean R2: {final_scores.mean():.4f}")
print(f"Folds R2: {final_scores}")


In [ ]:
# 21.1) Getting the MAE scores for the final XGB 

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
import numpy as np

# 1) PREPARE THE DATA AND STRATEGY
X_final = df_full[[
    "video_duration_seconds", "title_length", "year", 
    "is_live", "is_pomodoro", "has_visuals", "is_long_form",
    "log_subscribers", "log_channel_views", "channel_video_count"
]]
y_final = df_full["log_engagement"]
groups_final = df_full["keyword_used"]

gkfold = GroupKFold(n_splits=5)
manual_mae_scores = []

print("Starting Manual Group CV for MAE...")

# 2) MANUAL LOOP: This bypasses the cross_val_score metadata issues
for i, (train_idx, val_idx) in enumerate(gkfold.split(X_final, y_final, groups=groups_final)):
    
    # Split data for this fold
    X_train_fold, X_val_fold = X_final.iloc[train_idx], X_final.iloc[val_idx]
    y_train_fold, y_val_fold = y_final.iloc[train_idx], y_final.iloc[val_idx]
    
    # Initialize and fit the winner model
    # No need to pass 'groups' here because we already did the split manually
    model_fold = xgb.XGBRegressor(
        n_estimators=1000, 
        learning_rate=0.02, 
        max_depth=6, 
        random_state=42, 
        n_jobs=-1
    )
    
    model_fold.fit(X_train_fold, y_train_fold)
    
    # Predict and calculate MAE
    preds = model_fold.predict(X_val_fold)
    fold_mae = mean_absolute_error(y_val_fold, preds)
    
    manual_mae_scores.append(fold_mae)
    print(f"Fold {i+1} completed. MAE: {fold_mae:.4f}")

# 3) FINAL SUMMARY
mean_mae = np.mean(manual_mae_scores)
print("\n--- Final Ultimate MAE Results ---")
print(f"Mean CV MAE: {mean_mae:.4f}")
print(f"All Folds MAE: {manual_mae_scores}")


In [ ]:
# 21.2) Obtaining the actual MAE

# True MAE in real units (Likes + Comments)
true_mae = np.mean(np.abs(np.expm1(y_final) - np.expm1(xgb_final.predict(X_final))))

print(f"True MAE: {true_mae:.2f} interactions")

# On a log scale, the average error is ~1.03 units. This represents a high precision in "order of magnitude."

In [ ]:
# 22) Feature importance for the ultimate XGBoost model

import matplotlib.pyplot as plt
import xgboost as xgb

# 1) Fit the model on the full ultimate dataset
xgb_final.fit(X_final, y_final)

# 2) Get feature importance scores
# 'gain' shows which feature contributed most to the model's accuracy
importance_type = 'gain' 
scores = xgb_final.get_booster().get_score(importance_type=importance_type)

# 3) Convert to DataFrame for easier plotting
importance_df = pd.DataFrame({
    'Feature': list(scores.keys()),
    'Importance': list(scores.values())
}).sort_values(by='Importance', ascending=True)

# 4) Plotting
plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='teal')
plt.xlabel(f"Importance Score ({importance_type})")
plt.title("The Pillars of Engagement: XGBoost Feature Importance")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# Print the top features numerically
print("\n--- Feature Ranking ---")
print(importance_df.sort_values(by='Importance', ascending=False))


In [ ]:
# 23) Saving the final dataset with all features for future use (e.g., in a dashboard or further analysis)

# Save the final processed dataset to a CSV file
df_full.to_csv("youtube_niche_data_final.csv", index=False)
print("Dataset saved as: youtube_niche_data_final.csv")


In [ ]:
# 24) FINAL FIT & SAVING MODELS

import joblib

# A) Final training: We fit the model with ALL data (X_final, y_final)
# This is crucial before saving, otherwise, the model is "empty"
print("Training Ultimate XGBoost on full dataset...")
xgb_final.fit(X_final, y_final)

# B) Save the Ultimate XGBoost Model
joblib.dump(xgb_final, 'ultimate_xgb_model.pkl')

# C) Save the best Random Forest (Model 3) as a backup
# Assuming rf_log was your best stable RF
joblib.dump(rf_log, 'best_rf_model.pkl')

# D) Save the final Dataset to avoid API calls later
df_full.to_csv("youtube_niche_data_final.csv", index=False)

print("-" * 30)
print("Don't worry, David, models and dataset saved successfully!")
print("Files created: ultimate_xgb_model.pkl, best_rf_model.pkl, youtube_niche_data_final.csv")


#### 🚀 Data Science Project Highlights: YouTube Niche Engagement

**Honest Validation:** Transitioned from an inflated **0.49 R2** (data leakage) to a realistic **0.18 R2 using GroupKFold by keyword**, ensuring the model generalizes to new topics.

**Feature Engineering:** Created **niche-specific features** (Is_Pomodoro, Is_Live, Has_Visuals) that captured the unique logic of the Lofi/Study With Me ecosystem.

**Authority Power:** Integrated YouTube API data (Subscriber Count, Channel Views) to account for "Channel Authority," stabilizing the model significantly.

**Winning Algorithm:** Deployed XGBoost, achieving a powerful **0.37 Mean R2**, with specific niche folds reaching up to **0.83 R2**.

**Key Insight:** Proved that while channel size matters, content format (duration and specific title triggers) remains a critical driver for engagement.


In [ ]:
#25) Reloading the session for future analysis or dashboard integration
# ==========================================
# RELOAD SESSION: Lofi/Study Engagement Model
# ==========================================
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb

# 1) Load the dataset
df_full = pd.read_csv("youtube_niche_data_final.csv")

# 2) Load the winning model
xgb_final = joblib.load('ultimate_xgb_model.pkl')

# 3) Re-define the exact Feature List (MUST match the training order)
features_ultimate = [
    "video_duration_seconds", "title_length", "year", 
    "is_live", "is_pomodoro", "has_visuals", "is_long_form",
    "log_subscribers", "log_channel_views", "channel_video_count"
]

# 4) Re-create X and y for analysis
X_final = df_full[features_ultimate]
y_final = df_full["log_engagement"]

print("--- Session Loaded Successfully ---")
print(f"Dataset shape: {df_full.shape}")
print(f"Model ready to predict with {len(features_ultimate)} features.")


In [ ]:
# 26) SHAP for Ultimate XGBoost Model

import shap
import matplotlib.pyplot as plt

# 1) INITIALIZE THE SHAP EXPLAINER
# Using TreeExplainer, optimized for XGBoost
explainer = shap.TreeExplainer(xgb_final)

# 2) CALCULATE SHAP VALUES
# We use X_final (the same data the model was trained on)
shap_values = explainer.shap_values(X_final)

# 3) VISUALIZE: THE SUMMARY PLOT
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_final, plot_type="dot")


| Aspect                    | 🌳 Random Forest                       | ⚡ XGBoost                                         |
| ------------------------- | -------------------------------------- | ------------------------------------------------- |
| **Ensemble Method**       | Bagging (trees built independently)    | Boosting (trees built sequentially)               |
| **Learning Strategy**     | Averages many trees to reduce variance | Each tree corrects previous errors (reduces bias) |
| **Training Behavior**     | Parallel training                      | Sequential training                               |
| **Performance**           | Strong and stable baseline             | Often higher peak performance                     |
| **Hyperparameter Tuning** | Works well with minimal tuning         | Requires careful tuning                           |
| **Overfitting Risk**      | Lower risk out-of-the-box              | Higher risk if not tuned properly                 |
| **Computation Cost**      | Moderate                               | Higher                                            |
| **Interpretability**      | Easier to interpret                    | More complex (but SHAP-friendly)                  |
| **Best Use Case**         | Reliable baseline models               | Competitive / optimized ML tasks                  |


In [ ]:
# 27) Correlation Matrix for Numerical Features

import seaborn as sns
import matplotlib.pyplot as plt

# 1) PREPARE THE DATASET
# Using the same features for the Ultimate XGBoost
correlation_cols = [
    "log_engagement", "video_duration_seconds", "title_length", 
    "year", "is_live", "is_pomodoro", "is_long_form",
    "log_subscribers", "log_channel_views"
]

# Creating the correlation matrix (Pearson by default)
corr_matrix = df_full[correlation_cols].corr()

# 2) PLOT THE HEATMAP
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix, 
    annot=True, 
    fmt=".2f", 
    cmap='RdBu_r', # Red for positive, Blue for negative
    center=0,
    linewidths=0.5,
    cbar_kws={"shrink": .8}
)

plt.title("Ultimate Correlation Matrix: Niche & Authority vs Engagement")
plt.xticks(rotation=45, ha='right')
plt.show()

# 3) NUMERICAL SUMMARY
print("\n--- Direct Correlation with log_engagement ---")
print(corr_matrix["log_engagement"].sort_values(ascending=False))


In [ ]:
df_full.shape

### Dataset Structure Overview

#### **df_all – Semantic Layer**
- Raw dataset obtained directly from the YouTube Search API.
- Contains structured metadata per video (title, description, keyword).
- Preserves original data shape and API naming conventions.

---

#### **df_stats – Technical Layer**
- Enriched dataset from the YouTube Videos & Channels endpoints.
- Adds performance and authority metrics:
  - view_count, like_count, comment_count.
  - **subscriber_count, channel_view_count, channel_video_count** (Authority metrics).
- Bridges semantic content with both video performance and channel power.

---

#### **df_full – Final Project Dataset**
- Shape: **(925, 28)**
- Covers years: **2019 – 2026**
- Represents the complete analytical foundation of the project.

This dataset allows us to:
- Calculate **log-transformed** engagement for modeling stability.
- Extract niche-specific triggers (Pomodoro, Live, Long-form).
- Account for **Channel Authority** in predictive modeling.
- Train and compare high-performance models like **XGBoost**.
- Export a Tableau-ready dataset for advanced visualization.

---

### Engineered Variables

Additional derived features created for high-accuracy analysis:

- **log_engagement** – target variable (log1p of likes + comments).
- **is_pomodoro / is_live** – boolean triggers extracted from titles.
- **has_visuals** – detection of high-quality/4K content markers.
- **is_long_form** – binary flag for videos > 20 minutes.
- **log_subscribers** – log-transformed authority for linear scaling.
- **video_duration_seconds** – converted from ISO 8601 format.

---

### Dataset Summary (Key Metrics)

- Total videos: **925**
- Model Performance (Ultimate XGBoost): **0.37 Mean R2** (Best Fold: **0.83**)
- Top Engagement Drivers: **Channel Authority** and **Niche Specialization** (Pomodoro).
- Dataset spans from **2019 to 2026**.



#### 💡Top Predictive Insights (SHAP Analysis)

Based on the **SHAP Global Impact** and **XGBoost Feature Importance**, these are the 3 key drivers of engagement in the Lofi/Study niche:

1.  **The "Authority Floor" (Subscribers):** 
    - `log_subscribers` is the strongest predictor. It acts as a "guaranteed baseline" for engagement. However, the model proves that high authority alone doesn't guarantee a viral hit; it just sets a higher starting point.

2.  **Niche Specialization > General Content:** 
    - The presence of **"Pomodoro"** and **"Live"** triggers in titles significantly shifts engagement to the right (positive impact). Even channels with lower total views (**blue dots in SHAP**) achieve high engagement when these specific study-aid features are present.

3.  **Recency Bias (Temporal Trend):** 
    - The **"Year"** feature shows a strong positive correlation in recent years (2023-2024). This indicates the *Study With Me* niche is currently in a growth phase, with newer videos outperforming legacy content in terms of active user interaction.


In [ ]:
# 27) Distribution of videos per keyword (Filtered Dataset)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
df_full['keyword_used'].value_counts().plot(kind='bar', color='skyblue', edgecolor='black')

plt.title("Final Dataset: Number of videos per Keyword", fontsize=14)
plt.ylabel("Count of Videos", fontsize=12)
plt.xlabel("Search Keyword", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
import datetime
import pandas as pd
# 28) Yearly distribution of videos (Filtered Dataset)
# Ensure datetime conversion
df_full['published_at'] = pd.to_datetime(df_full['published_at'])
current_year = datetime.datetime.now().year

# Re-generating clean year labels
df_full['year'] = df_full['published_at'].dt.year
df_full['year_label'] = df_full['year'].apply(lambda x: f"{x} (YTD)" if x == current_year else str(x))

# Plotting with sorted index to avoid years jumping around
plt.figure(figsize=(10, 6))
df_full.groupby(['year', 'year_label']).size().reset_index(name='count').plot(
    kind='bar', x='year_label', y='count', ax=plt.gca(), color='teal', legend=False
)

plt.title(f"Content Volume per Year (Up to {current_year})", fontsize=14)
plt.ylabel("Number of Videos", fontsize=12)
plt.xlabel("Year of Publication", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.show()


In [ ]:
# 29) Niche evolution
# ==========================================
# NICHE EVOLUTION ANALYSIS (2019 - 2026)
# ==========================================
import matplotlib.pyplot as plt

# 1) DATA AGGREGATION
# We create a pivot table for the plot and a summary for the console
pivot_trends = (
    df_full[(df_full['year'] >= 2019) & (df_full['year'] <= 2026)]
    .groupby(['year', 'keyword_used'])
    .size()
    .unstack(fill_value=0)
)

# 2) CONSOLE INSIGHTS (Replacing the redundant prints)
print("--- Deep Dive: Year 2019 vs Niche Growth ---")
if 2019 in pivot_trends.index:
    print(f"Total videos in 2019: {pivot_trends.loc[2019].sum()}")
    print(pivot_trends.loc[2019][pivot_trends.loc[2019] > 0])
else:
    print("Year 2019 has 0 videos in the final dataset.")

# 3) EVOLUTION PLOT
plt.figure(figsize=(12, 6))
# Using the already calculated pivot_trends for the plot
pivot_trends.plot(kind='line', marker='o', linewidth=2.5, ax=plt.gca())

# Visual refinements
plt.title("Niche Explosion: Video Volume by Keyword (2019-2026)", fontsize=14, pad=15)
plt.ylabel("Number of Videos", fontsize=12)
plt.xlabel("Year", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(title="Keywords", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(range(2019, 2027)) # Ensure every year is represented
plt.tight_layout()
plt.show()

# 4) STATISTICAL EVOLUTION (2019 vs Peak Years)
# Quick comparison of how features evolved since the base year
comparison = df_full[df_full['year'].isin([2019, 2024, 2025])].groupby('year').agg({
    'video_id': 'count',
    'is_pomodoro': 'sum',
    'is_live': 'sum',
    'subscriber_count': 'mean'
}).rename(columns={'video_id': 'total_videos', 'subscriber_count': 'avg_subscribers'})

print("\n--- Key Metrics Evolution ---")
print(comparison)


### What is happening with the year 2019?

- Lower real volume in that year
- Lower indexing under those exact keywords
- Algorithm prioritizing other years
- Changes in naming trends
- Subsequent saturation (the niche exploded between 2020–2025)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
# 30) Authority vs Engagement: The Power Plot

# 1) THE POWER PLOT: Subscribers vs Engagement
# This is your #3 feature, much more important than title length
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=df_full, 
    x="log_subscribers", 
    y="log_engagement", 
    hue="keyword_used", 
    alpha=0.6,
    palette="viridis"
)

# 2) ADDING A TREND LINE
sns.regplot(data=df_full, x="log_subscribers", y="log_engagement", scatter=False, color='red')

plt.title("Authority vs. Success: How Subscriber Count Drives Engagement", fontsize=14)
plt.xlabel("Log Subscribers (Channel Power)", fontsize=12)
plt.ylabel("Log Engagement (Success)", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 31) The Underdog Analysis: Finding Hidden Gems in the Niche

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1) CALCULATE RESIDUALS (Engagement vs. Expected by Subscribers)
# We find videos where the real engagement is MUCH higher than what their sub count suggests
# We'll use a simple linear baseline for this specific "Underdog" search
from sklearn.linear_model import LinearRegression

model_baseline = LinearRegression()
model_baseline.fit(df_full[['log_subscribers']], df_full['log_engagement'])
df_full['expected_engagement'] = model_baseline.predict(df_full[['log_subscribers']])
df_full['underdog_score'] = df_full['log_engagement'] - df_full['expected_engagement']

# 2) IDENTIFY TOP 5 UNDERDOGS
# Sorting by the highest positive difference from expectation
top_5_underdogs = df_full.sort_values(by='underdog_score', ascending=False).head(5)

# 3) VISUALIZE THE "HALL OF FAME"
plt.figure(figsize=(12, 7))
sns.scatterplot(data=df_full, x="log_subscribers", y="log_engagement", alpha=0.3, color='gray')
sns.scatterplot(data=top_5_underdogs, x="log_subscribers", y="log_engagement", color='red', s=150, edgecolor='black', label='Top 5 Underdogs')

# Add labels for the underdogs
for i, txt in enumerate(top_5_underdogs['channel_title']):
    plt.annotate(txt, (top_5_underdogs['log_subscribers'].iloc[i], top_5_underdogs['log_engagement'].iloc[i]), 
                 xytext=(10,5), textcoords='offset points', fontsize=9, fontweight='bold')

plt.title("The Underdog Analysis: High Impact with Low Subscriber Count", fontsize=14)
plt.xlabel("Log Subscribers (Channel Size)", fontsize=12)
plt.ylabel("Log Engagement (Real Success)", fontsize=12)
plt.legend()
plt.show()

# 4) SHOW THEIR SECRETS
print("--- Top 5 Underdog Videos Analysis ---")
print(top_5_underdogs[['channel_title', 'title', 'is_pomodoro', 'is_live', 'video_duration_seconds', 'log_engagement']])


In [ ]:
# 32) Checking channels with the most subscribers in the dataset
# 1) TOP 10 LARGEST CHANNELS (By Authority)
top_authority = df_full[['channel_title', 'subscriber_count', 'channel_view_count']].drop_duplicates().sort_values(by='subscriber_count', ascending=False).head(10)

print("--- The Giants (Top Authority) ---")
print(top_authority)


In [ ]:
# 33) Exporting the final dataset with predictions for Tableau

import pandas as pd

# 1) GENERATE FINAL PREDICTIONS
# We use the trained Ultimate XGBoost to get the model's opinion on every video
df_full['predicted_log_engagement'] = xgb_final.predict(X_final)

# 2) REVERSE LOG TRANSFORM (Crucial for Tableau)
# Tableau is better with real numbers. Let's convert logs back to real Likes+Comments
df_full['predicted_engagement_real'] = np.expm1(df_full['predicted_log_engagement'])
df_full['actual_engagement_real'] = np.expm1(df_full['log_engagement'])

# 3) CALCULATE ERROR (Residuals)
# This allows you to visualize in Tableau where the model missed (Underdogs vs. Overrated)
df_full['prediction_error'] = df_full['actual_engagement_real'] - df_full['predicted_engagement_real']

# 4) SELECT COLUMNS FOR EXPORT
# We keep everything: Metadata + Niche Features + Authority + Predictions
cols_to_export = [
    'video_id', 'title', 'published_at', 'channel_title', 'keyword_used', 
    'year', 'video_duration_seconds', 'title_length',
    'is_live', 'is_pomodoro', 'has_visuals', 'is_long_form',
    'subscriber_count', 'channel_view_count', 'channel_video_count',
    'actual_engagement_real', 'predicted_engagement_real', 'prediction_error'
]

# 5) SAVE TO CSV
df_tableau = df_full[cols_to_export]
df_tableau.to_csv("youtube_analysis_for_tableau.csv", index=False, encoding='utf-8-sig')

print("-" * 30)
print("File ready for Tableau: youtube_analysis_for_tableau.csv")
print(f"Total rows exported: {len(df_tableau)}")
print("Tip: Use 'actual_engagement_real' for the main charts in Tableau!")


### 📊 Tableau Visualization Layer (Ultimate Version)

#### **1. Raw & Authority Fields (Channel Power)**
- **title / description** – Semantic content.
- **channel_title / keyword_used** – Category and source filters.
- **subscriber_count** – **NEW:** Total channel authority.
- **channel_view_count** – **NEW:** Total channel historical reach.
- **channel_video_count** – **NEW:** Channel experience/consistency.

#### **2. Performance & Engineered Metrics**
- **actual_units** – **NEW:** Sum of (likes + comments) in real numbers.
- **predicted_units** – **NEW:** What the XGBoost model expected for this video.
- **abs_error_units** – **NEW:** Difference between reality and prediction (Residuals).
- **video_duration_seconds** – Time-based numerical analysis.
- **year** – Temporal trend dimension.

#### **3. Niche Strategy Triggers (Boolean)**
- **is_pomodoro** – **NEW:** Impact of timer-based study content.
- **is_live** – **NEW:** Performance of 24/7 radio/live streams.
- **is_long_form** – **NEW:** Strategic duration flag (> 20 min).
- **has_visuals** – **NEW:** Impact of high-quality/4K aesthetic markers.

---

### 🚀 What this lets us build in Tableau instantly:

*   **The "Underdog" Scatter Plot:** `subscriber_count` vs. `actual_units`. Use `abs_error_units` for color to highlight videos that "hacked" the algorithm (like **EYM**).
*   **Strategy ROI:** Bar chart comparing `is_pomodoro` vs. Average Engagement. Does a timer actually increase likes?
*   **Model Accuracy Dashboard:** Side-by-side comparison of `actual_units` vs. `predicted_units` per Keyword.
*   **Authority vs. Content:** A dual-axis chart showing how much engagement comes from channel size vs. video-specific features.
*   **Niche Evolution:** Line chart of `video_count` by `keyword_used` from 2019 to 2026.
*   **Engagement Heatmap:** `video_duration_seconds` vs. `title_length` colored by `actual_units` to find the "Sweet Spot."

---

### 💡 Suggested Filters for Dashboard
- **Keyword / Year / Channel Title**
- **Is Pomodoro? (Yes/No)**
- **Channel Size Category** (Small < 10k, Medium 10k-500k, Giant > 500k)


In [ ]:
# CHECKPOINT
# ==========================================
# RELOAD SESSION: Ultimate Lofi/Study Model
# ==========================================
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
from sklearn.metrics import r2_score, mean_absolute_error

# 1) LOAD DATASET & MODEL
# Make sure these files are in your current working directory
df_full = pd.read_csv("youtube_niche_data_final.csv")
xgb_final = joblib.load('ultimate_xgb_model.pkl')

# 2) DEFINE ULTIMATE FEATURE LIST
# This order MUST be identical to the one used during training
features_ultimate = [
    "video_duration_seconds", "title_length", "year", 
    "is_live", "is_pomodoro", "has_visuals", "is_long_form",
    "log_subscribers", "log_channel_views", "channel_video_count"
]

# 3) RECONSTRUCT X AND Y
X_final = df_full[features_ultimate]
y_final = df_full["log_engagement"]

# 4) VALIDATION CHECK (To ensure everything loaded correctly)
y_pred_log = xgb_final.predict(X_final)
current_r2 = r2_score(y_final, y_pred_log)
current_mae = mean_absolute_error(y_final, y_pred_log)

print("-" * 30)
print("--- SESSION LOADED SUCCESSFULLY ---")
print(f"Dataset Shape: {df_full.shape}")
print(f"Model Features: {len(features_ultimate)}")
print(f"Verified R2 on full data: {current_r2:.4f}")
print(f"Verified Log-MAE: {current_mae:.4f}")
print("-" * 30)
print("Ready to continue, David!")


### Machine Learning - Final Conclusion

- **Platform Complexity:** Predicting YouTube engagement remains inherently complex due to hidden algorithmic exposure; however, we achieved a significant **0.37 Mean R2**, proving that a well-tuned model can capture substantial behavioral patterns.
- **Strategic Features:** Features specifically engineered for the lofi/study niche (**is_pomodoro, is_live**) were critical, showing that content strategy can act as a catalyst for engagement beyond raw metrics.
- **The Authority Factor:** Integrating channel-level metrics (**Subscribers and Channel Views**) was the "game-changer," nearly doubling predictive performance and confirming that **creator scale is the primary dominant factor**.
- **Model Supremacy:** While Random Forest provided a stable baseline, **XGBoost** emerged as the superior algorithm, effectively capturing "obsessive" patterns and corrections that simpler tree-based models missed.
- **Underdog Identification:** Residual analysis revealed **"Underdog" channels** (like EYM) that significantly outperform their expected engagement, proving that high-quality niche content can "hack" the authority-based algorithm.
- **Stability vs. Power:** Our final **GroupKFold** validation ensures that these results are not due to data leakage but represent a true ability to generalize across different search keywords.
- **Interpretability:** SHAP values confirmed that while **Year and Subscribers** set the floor for success, specific video attributes provide the "ceiling" for viral performance in the Lofi/Study ecosystem.


### We close the ML-Section of the project with:

- **Full-Spectrum API Extraction:** Integration of Video, Statistics, and Channel Authority data.
- **Advanced Data Cleaning:** Managing niche-specific outliers and temporal data.
- **Strategic Feature Engineering:** Development of niche triggers (*Pomodoro, Live, Long-form*).
- **Log-Stabilized Target Variable:** Refinement of `log_engagement` to handle viral skewness.
- **Model Evolution:** 
    - Baseline: **Linear Regression**
    - Non-linear Stability: **Random Forest**
    - High-Performance Boosting: **XGBoost** (Winning Model: **0.37 R2**)
- **Deep Model Interpretation:** Implementation of **SHAP Values** and **Feature Importance (Gain)**.
- **Ensemble Experiments:** Testing **Voting and Stacking Regressors** (RF + MLP).
- **Leakage Control:** Rigorous validation using **GroupKFold** by keyword to ensure real-world generalization.


#### Next-up: Binaural Beat Generator

### X) Binaural Beat Generator
### **“Mixing real music with controllable brainwave tones.”**

### BRAINWAVES / STATES OF MIND (Hertz - Hz)

These ranges (0.5–100 Hz) describe the **brain’s own electrical oscillations** as measured by EEG. They refer to electrical brain activity, measured in Hz — but not sound. They reflect your mental and emotional state (calm, focused, sleeping, etc.).



| Range (Hz)     | Brain State         | Description |
|----------------|---------------------|-------------|
| 0.5–4          | Delta               | Deep sleep  
| 4–8            | Theta               | Meditation, drowsiness  
| 8–12           | Alpha               | Relaxed alertness
| 12–30          | Beta                | Focus, thinking  
| 30–100         | Gamma               | High-level cognition, perception


***Remember:***
Brain’s **own** electrical oscillations, they happen inside the brain **regardless of music**.

In [ ]:
# Setting up libraries for binaural beats generation

import numpy as np
from scipy.io import wavfile
from scipy.io.wavfile import write
from IPython.display import Audio


In [ ]:
# Binaural, no music 

import numpy as np
from scipy.io.wavfile import write
from IPython.display import Audio

# ======================
# Parameters
# ======================
duration = 600.0   # seconds
fs = 44100        # sampling rate
freq_left = 100  # Hz
freq_right = 130 # Hz
volume = 0.05    # safe for ears

# ======================
# Time array
# ======================

t = np.linspace(0, duration, int(fs*duration), endpoint=False)

# ======================
# Generate sine waves for left and right ears
# ======================

left_tone = np.sin(2 * np.pi * freq_left * t)
right_tone = np.sin(2 * np.pi * freq_right * t)

# ======================
# Stereo signal (left, right)
# ======================

stereo_signal = np.vstack((left_tone, right_tone)).T * volume

# ======================
# Convert to int16 for WAV
# ======================

stereo_signal_int16 = (stereo_signal * 32767).astype(np.int16)

# ======================
# Save as WAV
# ======================
write("binaural_test.wav", fs, stereo_signal_int16)
print("✅ Exported binaural beat to binaural_test.wav")

# ======================
# Playable widget
# ======================

Audio("binaural_test.wav")


In [ ]:
import numpy as np
from scipy.io import wavfile
from scipy.io.wavfile import write
from IPython.display import Audio


# ======================
# Load background track
# ======================
fs_bg, music = wavfile.read("Velvet Chill Sounds - Track 242.wav")

# Convert to float range [-1, 1]
if music.dtype != np.float32 and music.dtype != np.float64:
    music = music.astype(np.float32) / np.max(np.abs(music))

# Ensure stereo
if music.ndim == 1:
    music = np.stack([music, music], axis=1)

# Use the background track sample rate
fs = fs_bg


# ======================
# Parameters
# ======================
duration = music.shape[0] / fs
freq_left = 100
freq_right = 120
volume = 0.05


# ======================
# Generate binaural tones
# ======================
t = np.linspace(0, duration, int(fs * duration), endpoint=False)

left_tone  = np.sin(2 * np.pi * freq_left  * t)
right_tone = np.sin(2 * np.pi * freq_right * t)

stereo_signal = np.stack((left_tone, right_tone), axis=1) * volume


# ======================
# Merge background and binaural tones
# ======================
length = min(len(music), len(stereo_signal))

music = music[:length]
stereo_signal = stereo_signal[:length]

merged = music + stereo_signal

# Prevent clipping
merged = np.clip(merged, -1.0, 1.0)


# ======================
# Export
# ======================
merged_int16 = (merged * 32767).astype(np.int16)

write("neurobeats_merge.wav", fs, merged_int16)

print("✅ Exported merged track to neurobeats_merge.wav")


# ======================
# Preview
# ======================
Audio("neurobeats_merge.wav")


In [ ]:
# Creating Offsets
import numpy as np
import wave

# ======================
# Parameters
# ======================

duration = 30 * 60     # 30 minutes
fs = 22050

base_freq = 100.0 # This is the base we hear, the difference between one frequency and another is what influences the mental state
volume = 0.05

output_file = "binaural_offset_30min_alpha_beta_gamma.wav"

block_size = fs * 10   # 10 seconds per block

# ======================
# Total samples
# ======================

total_samples = int(duration * fs)

# ======================
# Beat ramp with zones (Alpha / Beta / short Gamma)
# ======================

alpha_beat = 8.0
beta_beat  = 13.0
gamma_beat = 30.0

alpha_minutes = 10
beta_minutes  = 10
gamma_minutes = 2     # short gamma, long exposure is tiresome and can provoke headaches

transition1_minutes = 4   # alpha - beta
transition2_minutes = 4   # beta  - gamma

alpha_samples = int(alpha_minutes * 60 * fs)
beta_samples  = int(beta_minutes  * 60 * fs)
gamma_samples = int(gamma_minutes * 60 * fs)

transition1_samples = int(transition1_minutes * 60 * fs)
transition2_samples = int(transition2_minutes * 60 * fs)

used = (
    alpha_samples +
    transition1_samples +
    beta_samples +
    transition2_samples +
    gamma_samples
)

if used > total_samples:
    raise ValueError("Total duration is too short for the selected zones.")

# If there is any leftover time, extend the last gamma zone
rest = total_samples - used

# ---- Zone 1 : Alpha stable
zone1 = np.full(alpha_samples, alpha_beat, dtype=np.float32)

# ---- Transition 1 : Alpha -> Beta (smooth)
x1 = np.linspace(0, 1, transition1_samples, dtype=np.float32)
smooth1 = 0.5 - 0.5 * np.cos(np.pi * x1)
zone2 = alpha_beat + (beta_beat - alpha_beat) * smooth1

# ---- Zone 2 : Beta stable
zone3 = np.full(beta_samples, beta_beat, dtype=np.float32)

# ---- Transition 2 : Beta -> Gamma (smooth)
x2 = np.linspace(0, 1, transition2_samples, dtype=np.float32)
smooth2 = 0.5 - 0.5 * np.cos(np.pi * x2)
zone4 = beta_beat + (gamma_beat - beta_beat) * smooth2

# ---- Zone 3 : Gamma (short burst)
zone5 = np.full(gamma_samples, gamma_beat, dtype=np.float32)

# ---- Optional tail (keep last state if time remains)
zone6 = np.full(rest, gamma_beat, dtype=np.float32)

beat_ramp = np.concatenate(
    [zone1, zone2, zone3, zone4, zone5, zone6]
).astype(np.float32)


# ======================
# Phase accumulators + index
# ======================

left_phase = 0.0
right_phase = 0.0
write_index = 0

# ======================
# Stream write WAV
# ======================

with wave.open(output_file, "wb") as wf:
    wf.setnchannels(2)
    wf.setsampwidth(2)   # int16
    wf.setframerate(fs)

    while write_index < total_samples:

        n = min(block_size, total_samples - write_index)

        beat_block = beat_ramp[write_index:write_index + n].astype(np.float32)

        left_freq_block = np.full(n, base_freq, dtype=np.float32)
        right_freq_block = base_freq + beat_block

        left_phase_block = left_phase + 2*np.pi * np.cumsum(left_freq_block / fs)
        right_phase_block = right_phase + 2*np.pi * np.cumsum(right_freq_block / fs)

        left_block = np.sin(left_phase_block)
        right_block = np.sin(right_phase_block)

        stereo_block = np.stack([left_block, right_block], axis=1) * volume

        stereo_block = np.clip(stereo_block, -1.0, 1.0)
        stereo_block_int16 = (stereo_block * 32767).astype(np.int16)

        wf.writeframes(stereo_block_int16.tobytes())

        left_phase = float(left_phase_block[-1])
        right_phase = float(right_phase_block[-1])

        write_index += n

print("✅ Exported:", output_file)
